# Corporate Credit Risk Assessment

## Overview
This project implements a **multi-factor corporate credit assessment framework** designed to evaluate the financial health, default risk, and overall creditworthiness of a company.

The model combines **fundamental accounting metrics**, **market-based credit indicators**, and **structural default modeling** to generate a comprehensive credit risk profile and an implied internal credit rating.

---

## Model Included

### Corporate Creditworthiness Assessment Framework
The model evaluates company credit quality using five key methodologies commonly applied in **credit risk analysis**, **fixed income investing**, and **corporate lending**.

The framework integrates:

#### 1. Altman Z-Score
A bankruptcy prediction model based on balance sheet and profitability metrics.

The model classifies firms into:

- **Safe Zone** → low bankruptcy risk  
- **Grey Zone** → moderate financial risk  
- **Distress Zone** → elevated bankruptcy probability  

---

#### 2. Piotroski F-Score
A financial quality assessment model evaluating company fundamentals across:

- Profitability  
- Liquidity  
- Leverage  
- Operating efficiency  

Scores range from **0 to 9**, where higher values indicate stronger financial quality.

---

#### 3. Credit Spread / Option-Adjusted Spread (OAS)
A market-based measure of perceived credit risk calculated as the spread between:

- Corporate bond yield  
- Risk-free benchmark yield  

Higher spreads imply higher perceived default risk and weaker credit quality.

---

#### 4. Merton Distance-to-Default Model
A structural credit risk model estimating:

- Distance-to-default (DD)  
- Probability of default (PD)

using:

- Equity market value  
- Equity volatility  
- Debt obligations  
- Risk-free rate assumptions

---

#### 5. Expected Loss (EL)
A credit loss estimation framework using:

**Expected Loss = PD × LGD × EAD**

Where:

- **PD** → Probability of Default  
- **LGD** → Loss Given Default  
- **EAD** → Exposure at Default  

This model estimates the expected dollar loss associated with a credit exposure.

---

## Composite Credit Rating System
The framework combines all five methodologies into a **weighted internal credit rating model** producing:

- Composite credit score (0–100)  
- Implied rating (**AAA → D**)  
- Investment recommendation

This approach enables a more holistic evaluation of corporate creditworthiness beyond any single standalone metric.

---

## Financial Concepts Applied
- Credit Risk Analysis  
- Bankruptcy Prediction Models  
- Altman Z-Score  
- Piotroski F-Score  
- Option-Adjusted Spread (OAS)  
- Merton Structural Default Model  
- Probability of Default (PD)  
- Expected Loss Modeling  
- Internal Credit Rating Systems  

---

## Technologies Used
- Python  
- NumPy  
- Pandas  
- SciPy  

---

## Objective
The objective of this project is to build a practical **corporate credit assessment framework** capable of evaluating financial quality, default risk, and expected loss while generating an internally derived credit rating for investment and lending analysis.

# Corporate Credit Assessment Model

## Methodology

The model evaluates corporate creditworthiness using a combination of:

- Accounting-based risk measures  
- Market-implied credit indicators  
- Structural default probability models  
- Expected loss frameworks

The combined output generates a weighted internal credit rating designed to support investment and lending decisions.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import minimize

class CreditAssessmentModel:
    """
    Single company credit worthiness assessment model.
    Implements 5 key metrics: Altman Z-Score, Piotroski F-Score, 
    Credit Spread/OAS, Merton Distance-to-Default, and Expected Loss.
    """
    
    def __init__(self, company_name):
        self.company_name = company_name
        self.results = {}
        
    def altman_z_score(self, working_capital, total_assets, retained_earnings, 
                       ebit, market_value_equity, total_liabilities, sales):
        """
        Calculate Altman Z-Score for bankruptcy prediction.
        
        Z > 2.99: Safe zone
        1.81 < Z < 2.99: Grey zone
        Z < 1.81: Distress zone
        
        Parameters:
        - working_capital: Current Assets - Current Liabilities
        - total_assets: Total Assets
        - retained_earnings: Retained Earnings
        - ebit: Earnings Before Interest and Taxes
        - market_value_equity: Market Cap
        - total_liabilities: Total Liabilities
        - sales: Revenue/Sales
        """
        X1 = working_capital / total_assets
        X2 = retained_earnings / total_assets
        X3 = ebit / total_assets
        X4 = market_value_equity / total_liabilities
        X5 = sales / total_assets
        
        z_score = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
        
        if z_score > 2.99:
            interpretation = "Safe Zone - Low bankruptcy risk"
        elif z_score > 1.81:
            interpretation = "Grey Zone - Moderate risk"
        else:
            interpretation = "Distress Zone - High bankruptcy risk"
            
        self.results['Altman Z-Score'] = {
            'value': z_score,
            'interpretation': interpretation
        }
        return z_score, interpretation
    
    def piotroski_f_score(self, net_income, operating_cashflow, roa_current, roa_prior,
                          total_assets_current, total_assets_prior, long_term_debt_current,
                          long_term_debt_prior, current_ratio_current, current_ratio_prior,
                          shares_outstanding_current, shares_outstanding_prior,
                          gross_margin_current, gross_margin_prior, asset_turnover_current,
                          asset_turnover_prior):
        """
        Calculate Piotroski F-Score (0-9 scale).
        
        Score >= 7: Strong financial position
        Score 4-6: Average
        Score <= 3: Weak financial position
        
        Parameters for 9 binary signals (each worth 1 point):
        Profitability (4):
        - net_income > 0
        - operating_cashflow > 0
        - roa_current > roa_prior
        - operating_cashflow > net_income (quality of earnings)
        
        Leverage/Liquidity (3):
        - long_term_debt_current < long_term_debt_prior
        - current_ratio_current > current_ratio_prior
        - shares_outstanding_current <= shares_outstanding_prior (no dilution)
        
        Operating Efficiency (2):
        - gross_margin_current > gross_margin_prior
        - asset_turnover_current > asset_turnover_prior
        """
        score = 0
        
        # Profitability signals
        if net_income > 0:
            score += 1
        if operating_cashflow > 0:
            score += 1
        if roa_current > roa_prior:
            score += 1
        if operating_cashflow > net_income:
            score += 1
            
        # Leverage/Liquidity signals
        if long_term_debt_current < long_term_debt_prior:
            score += 1
        if current_ratio_current > current_ratio_prior:
            score += 1
        if shares_outstanding_current <= shares_outstanding_prior:
            score += 1
            
        # Operating Efficiency signals
        if gross_margin_current > gross_margin_prior:
            score += 1
        if asset_turnover_current > asset_turnover_prior:
            score += 1
            
        if score >= 7:
            interpretation = "Strong - High quality financial position"
        elif score >= 4:
            interpretation = "Average - Moderate financial quality"
        else:
            interpretation = "Weak - Poor financial position"
            
        self.results['Piotroski F-Score'] = {
            'value': score,
            'interpretation': interpretation
        }
        return score, interpretation
    
    def credit_spread_oas(self, corporate_bond_yield, risk_free_rate):
        """
        Calculate Option-Adjusted Spread (simplified as credit spread).
        
        Higher spread = Higher perceived credit risk
        
        Typical spreads:
        < 100 bps: Investment grade (AAA-BBB)
        100-300 bps: Lower investment grade
        300-500 bps: High yield (BB)
        > 500 bps: Distressed (B and below)
        
        Parameters:
        - corporate_bond_yield: Yield on company's bond (%)
        - risk_free_rate: Treasury yield of same maturity (%)
        """
        spread_bps = (corporate_bond_yield - risk_free_rate) * 100  # Convert to basis points
        
        if spread_bps < 100:
            interpretation = "Strong credit quality (Investment Grade)"
        elif spread_bps < 300:
            interpretation = "Moderate credit quality (Lower IG)"
        elif spread_bps < 500:
            interpretation = "Speculative quality (High Yield)"
        else:
            interpretation = "High risk (Distressed)"
            
        self.results['Credit Spread (OAS)'] = {
            'value': spread_bps,
            'interpretation': interpretation,
            'unit': 'basis points'
        }
        return spread_bps, interpretation
    
    def merton_distance_to_default(self, market_value_equity, equity_volatility,
                                   total_debt, risk_free_rate, time_horizon=1):
        """
        Calculate Merton Distance-to-Default and implied Probability of Default.
        
        DD > 3: Very low default risk
        2 < DD < 3: Low risk
        1 < DD < 2: Moderate risk
        DD < 1: High default risk
        
        Parameters:
        - market_value_equity: Market cap ($)
        - equity_volatility: Annual equity volatility (decimal, e.g., 0.30 for 30%)
        - total_debt: Book value of total debt ($)
        - risk_free_rate: Risk-free rate (decimal, e.g., 0.05 for 5%)
        - time_horizon: Time horizon in years (default 1)
        """
        V = market_value_equity  # Initial guess for asset value
        sigma_V = equity_volatility  # Initial guess for asset volatility
        D = total_debt
        r = risk_free_rate
        T = time_horizon
        
        # Iterative solution for asset value and volatility (simplified)
        # Using approximation: V_A ≈ V_E + D, σ_A ≈ σ_E * V_E / (V_E + D)
        asset_value = market_value_equity + total_debt
        asset_volatility = equity_volatility * (market_value_equity / asset_value)
        
        # Calculate d2 (distance to default)
        d2 = (np.log(asset_value / D) + (r - 0.5 * asset_volatility**2) * T) / \
             (asset_volatility * np.sqrt(T))
        
        # Probability of default
        pd = norm.cdf(-d2)
        
        if d2 > 3:
            interpretation = "Very Low default risk"
        elif d2 > 2:
            interpretation = "Low default risk"
        elif d2 > 1:
            interpretation = "Moderate default risk"
        else:
            interpretation = "High default risk"
            
        self.results['Merton Distance-to-Default'] = {
            'distance': d2,
            'probability_of_default': pd,
            'interpretation': interpretation
        }
        return d2, pd, interpretation
    
    def expected_loss(self, probability_of_default, loss_given_default, exposure_at_default):
        """
        Calculate Expected Loss.
        
        EL = PD × LGD × EAD
        
        Parameters:
        - probability_of_default: PD (decimal, e.g., 0.05 for 5%)
        - loss_given_default: LGD (decimal, e.g., 0.45 for 45%)
        - exposure_at_default: EAD in dollars (your exposure to the company's debt)
        """
        expected_loss = probability_of_default * loss_given_default * exposure_at_default
        
        el_percentage = (expected_loss / exposure_at_default) * 100
        
        self.results['Expected Loss'] = {
            'value': expected_loss,
            'percentage': el_percentage,
            'pd': probability_of_default,
            'lgd': loss_given_default,
            'ead': exposure_at_default,
            'unit': 'USD'
        }
        return expected_loss
    
    def _score_metrics(self):
        """
        Score each metric on a 0-100 scale and calculate composite score.
        Higher score = Better credit quality
        """
        scores = {}
        
        # 1. Altman Z-Score (0-100 scale)
        if 'Altman Z-Score' in self.results:
            z = self.results['Altman Z-Score']['value']
            if z >= 2.99:
                scores['Altman'] = 100
            elif z >= 1.81:
                # Linear interpolation in grey zone
                scores['Altman'] = 50 + ((z - 1.81) / (2.99 - 1.81)) * 50
            else:
                # Linear from 0 at z=0 to 50 at z=1.81
                scores['Altman'] = max(0, (z / 1.81) * 50)
        
        # 2. Piotroski F-Score (0-100 scale)
        if 'Piotroski F-Score' in self.results:
            f = self.results['Piotroski F-Score']['value']
            scores['Piotroski'] = (f / 9) * 100
        
        # 3. Credit Spread (0-100 scale, inverted since lower is better)
        if 'Credit Spread (OAS)' in self.results:
            spread = self.results['Credit Spread (OAS)']['value']
            if spread <= 100:
                scores['Spread'] = 100
            elif spread <= 300:
                scores['Spread'] = 100 - ((spread - 100) / 200) * 30
            elif spread <= 500:
                scores['Spread'] = 70 - ((spread - 300) / 200) * 40
            else:
                scores['Spread'] = max(0, 30 - ((spread - 500) / 500) * 30)
        
        # 4. Merton DD (0-100 scale)
        if 'Merton Distance-to-Default' in self.results:
            dd = self.results['Merton Distance-to-Default']['distance']
            if dd >= 3:
                scores['Merton'] = 100
            elif dd >= 2:
                scores['Merton'] = 70 + ((dd - 2) / 1) * 30
            elif dd >= 1:
                scores['Merton'] = 40 + ((dd - 1) / 1) * 30
            else:
                scores['Merton'] = max(0, dd * 40)
        
        # 5. Expected Loss (0-100 scale, inverted since lower is better)
        if 'Expected Loss' in self.results:
            el_pct = self.results['Expected Loss']['percentage']
            if el_pct <= 1:
                scores['ExpectedLoss'] = 100
            elif el_pct <= 3:
                scores['ExpectedLoss'] = 100 - ((el_pct - 1) / 2) * 30
            elif el_pct <= 5:
                scores['ExpectedLoss'] = 70 - ((el_pct - 3) / 2) * 40
            else:
                scores['ExpectedLoss'] = max(0, 30 - ((el_pct - 5) / 5) * 30)
        
        return scores
    
    def get_composite_rating(self):
        """
        Calculate composite credit rating based on all metrics.
        Returns rating (AAA to D), score (0-100), and recommendation.
        """
        scores = self._score_metrics()
        
        if not scores:
            return None, None, "Insufficient data for rating"
        
        # Weighted average !! can be adjusted to give more weight on different lines
        weights = {
            'Altman': 0.25,      # Fundamental bankruptcy risk
            'Piotroski': 0.20,   # Financial quality
            'Spread': 0.20,      # Market perception
            'Merton': 0.25,      # Probability of default
            'ExpectedLoss': 0.10 # Dollar risk (lower weight as it depends on exposure)
        }
        
        composite_score = sum(scores.get(metric, 50) * weight 
                             for metric, weight in weights.items() 
                             if metric in scores)
        
        # Normalize if not all metrics present
        total_weight = sum(weights[metric] for metric in scores.keys())
        composite_score = composite_score / total_weight if total_weight > 0 else 50
        
        # Assign credit rating
        if composite_score >= 90:
            rating = "AAA"
            recommendation = "STRONG BUY - Excellent credit quality, minimal risk"
        elif composite_score >= 80:
            rating = "AA"
            recommendation = "BUY - Very good credit quality, low risk"
        elif composite_score >= 70:
            rating = "A"
            recommendation = "BUY - Good credit quality, low to moderate risk"
        elif composite_score >= 60:
            rating = "BBB"
            recommendation = "HOLD - Adequate credit quality, moderate risk"
        elif composite_score >= 50:
            rating = "BB"
            recommendation = "HOLD/CAUTION - Speculative, elevated risk"
        elif composite_score >= 40:
            rating = "B"
            recommendation = "SELL - Highly speculative, significant risk"
        elif composite_score >= 30:
            rating = "CCC"
            recommendation = "STRONG SELL - Substantial credit risk"
        else:
            rating = "D"
            recommendation = "AVOID - Extremely high default risk"
        
        self.results['Composite Rating'] = {
            'rating': rating,
            'score': composite_score,
            'recommendation': recommendation,
            'individual_scores': scores
        }
        
        return rating, composite_score, recommendation
    
    def generate_report(self):
        """Generate a comprehensive credit assessment report."""
        print(f"\n{'='*60}")
        print(f"CREDIT ASSESSMENT REPORT: {self.company_name}")
        print(f"{'='*60}\n")
        
        if 'Altman Z-Score' in self.results:
            print(f"1. ALTMAN Z-SCORE: {self.results['Altman Z-Score']['value']:.2f}")
            print(f"   → {self.results['Altman Z-Score']['interpretation']}\n")
        
        if 'Piotroski F-Score' in self.results:
            print(f"2. PIOTROSKI F-SCORE: {self.results['Piotroski F-Score']['value']}/9")
            print(f"   → {self.results['Piotroski F-Score']['interpretation']}\n")
        
        if 'Credit Spread (OAS)' in self.results:
            print(f"3. CREDIT SPREAD (OAS): {self.results['Credit Spread (OAS)']['value']:.0f} bps")
            print(f"   → {self.results['Credit Spread (OAS)']['interpretation']}\n")
        
        if 'Merton Distance-to-Default' in self.results:
            dd = self.results['Merton Distance-to-Default']
            print(f"4. MERTON DISTANCE-TO-DEFAULT: {dd['distance']:.2f}")
            print(f"   Probability of Default: {dd['probability_of_default']:.2%}")
            print(f"   → {dd['interpretation']}\n")
        
        if 'Expected Loss' in self.results:
            el = self.results['Expected Loss']
            print(f"5. EXPECTED LOSS: ${el['value']:,.2f}")
            print(f"   ({el['percentage']:.2f}% of exposure)")
            print(f"   PD: {el['pd']:.2%} | LGD: {el['lgd']:.2%} | EAD: ${el['ead']:,.2f}\n")
        
        print(f"{'='*60}")
        
        # Get and display composite rating
        rating, score, recommendation = self.get_composite_rating()
        
        if rating:
            print(f"\n📊 COMPOSITE CREDIT RATING")
            print(f"{'='*60}")
            print(f"\n   Rating: {rating}")
            print(f"   Composite Score: {score:.1f}/100")
            print(f"\n   RECOMMENDATION: {recommendation}")
            
            if 'Composite Rating' in self.results:
                print(f"\n   Individual Metric Scores (0-100):")
                for metric, metric_score in self.results['Composite Rating']['individual_scores'].items():
                    print(f"   • {metric}: {metric_score:.1f}")
            
            print(f"\n{'='*60}")
        
        print()


# Example Usage with AAPL
if __name__ == "__main__":
    # Initialize model
    model = CreditAssessmentModel("Example Corp")
    
    # 1. Altman Z-Score
    # Manual inputs from financial statements
    z_score, z_interp = model.altman_z_score(
        working_capital=17_674_000,      # Current Assets - Current Liabilities
        total_assets=359_241_000,       # Total Assets
        retained_earnings=-14_264_000,    # Retained Earnings
        ebit=133_050_000,                 # EBIT
        market_value_equity=4_225_152, # Market Cap
        total_liabilities=285_508_000,  # Total Liabilities
        sales=416_161_000               # Revenue
    )
    
    # 2. Piotroski F-Score
    f_score, f_interp = model.piotroski_f_score(
        net_income=112_010_000,           # Current year net income
        operating_cashflow=111_482_000,   # Current year OCF
        roa_current=0.757,                 # Current ROA - Net income/Total assets
        roa_prior=0.61,                   # Prior year ROA
        total_assets_current=147_957_000,
        total_assets_prior=152_987_000,
        long_term_debt_current=78_328_000,
        long_term_debt_prior=85_750_000,
        current_ratio_current=0.8933,        # Current Assets / Current Liabilities
        current_ratio_prior=0.8673,
        shares_outstanding_current=14_773_260,
        shares_outstanding_prior=15_116_786,
        gross_margin_current=0.4690,        # Gross Profit / Revenue
        gross_margin_prior=0.4620,
        asset_turnover_current=1.158,      # Sales / Total Assets
        asset_turnover_prior=1.0714,
    )
    
    # 3. Credit Spread (OAS)
    spread, spread_interp = model.credit_spread_oas(
        corporate_bond_yield=4.39,         # Company's bond yield (%) - Apple Inc. 4.75% 12-MAY-2035 - AAPL6070271
        risk_free_rate=4.14                # Treasury yield (%)      - 10 yrs bond
    )
    
    # 4. Merton Distance-to-Default
    dd, pd, dd_interp = model.merton_distance_to_default(
        market_value_equity=4_225_152,  # Market cap
        equity_volatility=0.3274,             # Equity volatility 1 yr - look equity volatility model 
        total_debt=285_508_000,           # Total debt
        risk_free_rate=0.0414,                # 4.14%
        time_horizon=1                     # 1 year
    )
    
    # 5. Expected Loss
    # Using PD from Merton model, assume LGD=45%, and your exposure
    el = model.expected_loss(
        probability_of_default=pd,        # From Merton model
        loss_given_default=0.45,          # 45% LGD assumption
        exposure_at_default=10_000_000    # Your exposure: $10M
    )
    
    # Generate comprehensive report
    model.generate_report()


CREDIT ASSESSMENT REPORT: Example Corp

1. ALTMAN Z-SCORE: 2.39
   → Grey Zone - Moderate risk

2. PIOTROSKI F-SCORE: 8/9
   → Strong - High quality financial position

3. CREDIT SPREAD (OAS): 25 bps
   → Strong credit quality (Investment Grade)

4. MERTON DISTANCE-TO-DEFAULT: 11.75
   Probability of Default: 0.00%
   → Very Low default risk

5. EXPECTED LOSS: $0.00
   (0.00% of exposure)
   PD: 0.00% | LGD: 45.00% | EAD: $10,000,000.00


📊 COMPOSITE CREDIT RATING

   Rating: AAA
   Composite Score: 91.5/100

   RECOMMENDATION: STRONG BUY - Excellent credit quality, minimal risk

   Individual Metric Scores (0-100):
   • Altman: 74.7
   • Piotroski: 88.9
   • Spread: 100.0
   • Merton: 100.0
   • ExpectedLoss: 100.0


